In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('35_built_in_functions').getOrCreate()

df = spark.read.csv("/FileStore/tables/sample_customer_data.csv", header=True, inferSchema=True)
df.show(5)

+----------+------------+----------+--------+-------+--------+------------+
|CustomerID|CustomerName|   Product|Quantity|  Price|Discount|PurchaseDate|
+----------+------------+----------+--------+-------+--------+------------+
|      1040|       Frank|Headphones|     5.0|30000.0|    NULL|  2024-01-01|
|      1007|       Diana|     Phone|     4.0|40000.0|    NULL|  2024-01-02|
|      1001|       Heidi|      NULL|     5.0|40000.0|     0.0|  2024-01-03|
|      1047|       Grace|     Phone|    NULL|40000.0|    15.0|  2024-01-04|
|      1017|       Heidi|    Laptop|     2.0|20000.0|    NULL|  2024-01-05|
+----------+------------+----------+--------+-------+--------+------------+
only showing top 5 rows



In [0]:
# 1. rename a column -> Only updates label(metadata)

df_1 = df.withColumnRenamed('CustomerName','c_name')

# rename multiple columns at a time
df_1 = df_1.withColumnRenamed('CustomerID','c_id')\
           .withColumnRenamed('Price','cost')
df_1.show(5)

+----+------+----------+--------+-------+--------+------------+
|c_id|c_name|   Product|Quantity|   cost|Discount|PurchaseDate|
+----+------+----------+--------+-------+--------+------------+
|1040| Frank|Headphones|     5.0|30000.0|    NULL|  2024-01-01|
|1007| Diana|     Phone|     4.0|40000.0|    NULL|  2024-01-02|
|1001| Heidi|      NULL|     5.0|40000.0|     0.0|  2024-01-03|
|1047| Grace|     Phone|    NULL|40000.0|    15.0|  2024-01-04|
|1017| Heidi|    Laptop|     2.0|20000.0|    NULL|  2024-01-05|
+----+------+----------+--------+-------+--------+------------+
only showing top 5 rows



In [0]:
#2. New or replace column

# If column_name already exists in df it updates, else creates new column.

from pyspark.sql.functions import col
df_2 = df.fillna({'Discount':0})

df_2 = df_2.withColumn('final_price',col('Price')-col('Discount')) \
           .withColumn('Quantity',col('Quantity')+1.0)
df_2.show(5)

+----------+------------+----------+--------+-------+--------+------------+-----------+
|CustomerID|CustomerName|   Product|Quantity|  Price|Discount|PurchaseDate|final_price|
+----------+------------+----------+--------+-------+--------+------------+-----------+
|      1040|       Frank|Headphones|     6.0|30000.0|     0.0|  2024-01-01|    30000.0|
|      1007|       Diana|     Phone|     5.0|40000.0|     0.0|  2024-01-02|    40000.0|
|      1001|       Heidi|      NULL|     6.0|40000.0|     0.0|  2024-01-03|    40000.0|
|      1047|       Grace|     Phone|    NULL|40000.0|    15.0|  2024-01-04|    39985.0|
|      1017|       Heidi|    Laptop|     3.0|20000.0|     0.0|  2024-01-05|    20000.0|
+----------+------------+----------+--------+-------+--------+------------+-----------+
only showing top 5 rows



In [0]:
#3. Remove a column  ->  Pass columns directly but not as list of columns

df_3 = df.drop('Quantity','Discount')
df_3.show(5)

+----------+------------+----------+-------+------------+
|CustomerID|CustomerName|   Product|  Price|PurchaseDate|
+----------+------------+----------+-------+------------+
|      1040|       Frank|Headphones|30000.0|  2024-01-01|
|      1007|       Diana|     Phone|40000.0|  2024-01-02|
|      1001|       Heidi|      NULL|40000.0|  2024-01-03|
|      1047|       Grace|     Phone|40000.0|  2024-01-04|
|      1017|       Heidi|    Laptop|20000.0|  2024-01-05|
+----------+------------+----------+-------+------------+
only showing top 5 rows



In [0]:
#4 . select()

df_4 = df.select('CustomerName','Price')
df_4.show(5)

+------------+-------+
|CustomerName|  Price|
+------------+-------+
|       Frank|30000.0|
|       Diana|40000.0|
|       Heidi|40000.0|
|       Grace|40000.0|
|       Heidi|20000.0|
+------------+-------+
only showing top 5 rows



In [0]:
#5. filter

# While combining multiple conditions inside .filter() or .where() we must use PARANTHESIS for each condition
df_5 = df.filter((df['Price']>39000) & (df['Quantity']>2))
df_5.show(5)

''' NOTE :- .filter() and .where() works same but
            .filter() works both on DataFrames and RDD's
            .where() works only on DataFrames but not on RDD's'''

+----------+------------+-------+--------+-------+--------+------------+
|CustomerID|CustomerName|Product|Quantity|  Price|Discount|PurchaseDate|
+----------+------------+-------+--------+-------+--------+------------+
|      1007|       Diana|  Phone|     4.0|40000.0|    NULL|  2024-01-02|
|      1001|       Heidi|   NULL|     5.0|40000.0|     0.0|  2024-01-03|
|      1045|         Eve|Monitor|     4.0|40000.0|     0.0|  2024-01-27|
|      1048|       Alice|  Phone|     3.0|40000.0|     0.0|  2024-02-06|
|      1021|       Alice|  Phone|     3.0|40000.0|    15.0|  2024-02-15|
+----------+------------+-------+--------+-------+--------+------------+
only showing top 5 rows



In [0]:
#6. orderBy

df_6 = df.orderBy(col('Quantity').desc(),col('Price').desc())
df_6.show(5)

+----------+------------+----------+--------+-------+--------+------------+
|CustomerID|CustomerName|   Product|Quantity|  Price|Discount|PurchaseDate|
+----------+------------+----------+--------+-------+--------+------------+
|      1005|     Charlie|      NULL|     5.0|40000.0|    15.0|  2024-03-01|
|      1001|       Heidi|      NULL|     5.0|40000.0|     0.0|  2024-01-03|
|      1023|         Bob|Headphones|     5.0|40000.0|    15.0|  2024-03-25|
|      1015|         Bob|   Monitor|     5.0|40000.0|     5.0|  2024-04-08|
|      1020|       Grace|    Tablet|     5.0|40000.0|     0.0|  2024-04-18|
+----------+------------+----------+--------+-------+--------+------------+
only showing top 5 rows



In [0]:
#7. drop duplicates -> Keeps one row, but dont delete all

df_7 = df.dropDuplicates(['CustomerID','CustomerName'])
df_7.count()

110

In [0]:
# union combines two dataframes -> retains duplicates
# union all is depricated in pyspark

In [0]:
# when() is similar to CASE WHEN THEN ELSE END in SQL

from pyspark.sql.functions import when, col

df_8 = df.withColumn(
    "Category",
    when(col("Price") >= 39000, "cat_1")
    .when((col("Price") > 30000) & (col("Price") < 39000), "cat_2")
    .otherwise("cat_3")
)

df_8.show(5)


+----------+------------+----------+--------+-------+--------+------------+--------+
|CustomerID|CustomerName|   Product|Quantity|  Price|Discount|PurchaseDate|Category|
+----------+------------+----------+--------+-------+--------+------------+--------+
|      1040|       Frank|Headphones|     5.0|30000.0|    NULL|  2024-01-01|   cat_3|
|      1007|       Diana|     Phone|     4.0|40000.0|    NULL|  2024-01-02|   cat_1|
|      1001|       Heidi|      NULL|     5.0|40000.0|     0.0|  2024-01-03|   cat_1|
|      1047|       Grace|     Phone|    NULL|40000.0|    15.0|  2024-01-04|   cat_1|
|      1017|       Heidi|    Laptop|     2.0|20000.0|    NULL|  2024-01-05|   cat_3|
+----------+------------+----------+--------+-------+--------+------------+--------+
only showing top 5 rows



In [0]:
# .contains() -> similar to LIKE '%ABC%' in SQL
# .startsWith() -> similar to LIKE 'ABC%' in SQL
# .endsWith() -> similar to LIKE '%ABC' in SQL

df_9 = df.filter(col('CustomerName').contains('an'))
df_9.show(3)
df_9.count()

+----------+------------+----------+--------+-------+--------+------------+
|CustomerID|CustomerName|   Product|Quantity|  Price|Discount|PurchaseDate|
+----------+------------+----------+--------+-------+--------+------------+
|      1040|       Frank|Headphones|     5.0|30000.0|    NULL|  2024-01-01|
|      1007|       Diana|     Phone|     4.0|40000.0|    NULL|  2024-01-02|
|      1047|       Diana|    Tablet|     4.0|10000.0|    15.0|  2024-01-09|
+----------+------------+----------+--------+-------+--------+------------+
only showing top 3 rows



31

In [0]:
# .describe() gives basic summary stats of numerical columns sometimes string columns too.

df.describe().show()

+-------+------------------+------------+----------+------------------+------------------+-----------------+
|summary|        CustomerID|CustomerName|   Product|          Quantity|             Price|         Discount|
+-------+------------------+------------+----------+------------------+------------------+-----------------+
|  count|               120|         120|       109|                96|                99|               97|
|   mean|1023.8583333333333|        NULL|      NULL|3.0833333333333335|27777.777777777777|7.835051546391752|
| stddev|15.032902755825535|        NULL|      NULL| 1.350763267011941|11298.867159718733|5.678322104671293|
|    min|              1000|       Alice|Headphones|               1.0|           10000.0|              0.0|
|    max|              1049|       Heidi|    Tablet|               5.0|           40000.0|             15.0|
+-------+------------------+------------+----------+------------------+------------------+-----------------+



In [0]:
# trim(col_name) removes leading and trailing whitespaces from string column
# ltrim removes leading only
# rtrim removes trailing only

from pyspark.sql.functions import trim

df_10 = df.withColumn('Updated_c_name', trim(col('CustomerName')))
df_10.show(5)


+----------+------------+----------+--------+-------+--------+------------+--------------+
|CustomerID|CustomerName|   Product|Quantity|  Price|Discount|PurchaseDate|Updated_c_name|
+----------+------------+----------+--------+-------+--------+------------+--------------+
|      1040|       Frank|Headphones|     5.0|30000.0|    NULL|  2024-01-01|         Frank|
|      1007|       Diana|     Phone|     4.0|40000.0|    NULL|  2024-01-02|         Diana|
|      1001|       Heidi|      NULL|     5.0|40000.0|     0.0|  2024-01-03|         Heidi|
|      1047|       Grace|     Phone|    NULL|40000.0|    15.0|  2024-01-04|         Grace|
|      1017|       Heidi|    Laptop|     2.0|20000.0|    NULL|  2024-01-05|         Heidi|
+----------+------------+----------+--------+-------+--------+------------+--------------+
only showing top 5 rows



In [0]:
# join -> df_1.join(df_2, on='col_name', how='inner')

In [0]:
# groupBy

from pyspark.sql.functions import sum

df_11 = df.groupBy('CustomerID').agg(sum(col('Quantity')).alias('total_quantity'))
df_11.show(5)

+----------+--------------+
|CustomerID|total_quantity|
+----------+--------------+
|      1025|           2.0|
|      1005|          12.0|
|      1016|           1.0|
|      1034|           5.0|
|      1046|           7.0|
+----------+--------------+
only showing top 5 rows



In [0]:
df.count()

120

In [0]:
# rank(), row_number(), dense_rank() WF

from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

window_spec = Window.partitionBy('Product').orderBy('Price')
df_12 = df.withColumn('ranking', dense_rank().over(window_spec))
df_12.show()

+----------+------------+----------+--------+-------+--------+------------+-------+
|CustomerID|CustomerName|   Product|Quantity|  Price|Discount|PurchaseDate|ranking|
+----------+------------+----------+--------+-------+--------+------------+-------+
|      1043|       Grace|      NULL|     2.0|   NULL|    15.0|  2024-01-11|      1|
|      1047|       Grace|      NULL|    NULL|   NULL|     5.0|  2024-01-12|      1|
|      1001|       Grace|      NULL|     4.0|   NULL|    10.0|  2024-01-24|      1|
|      1029|         Bob|      NULL|     3.0|   NULL|     0.0|  2024-02-26|      1|
|      1014|     Charlie|      NULL|     1.0|10000.0|    15.0|  2024-01-21|      2|
|      1037|       Grace|      NULL|     2.0|20000.0|     0.0|  2024-02-03|      3|
|      1014|         Bob|      NULL|     5.0|30000.0|    NULL|  2024-03-13|      4|
|      1029|       Grace|      NULL|     2.0|30000.0|    10.0|  2024-04-10|      4|
|      1001|       Heidi|      NULL|     5.0|40000.0|     0.0|  2024-01-03| 

In [0]:
# cast -> convert data type of column from one type to another

# for the given df lets convert purchasedate from date to string and customerId from int to string

df_13 = df.withColumn('CustomerID', col('CustomerID').cast('string'))\
          .withColumn('purchaseDatestring', col('PurchaseDate').cast('string'))

df_13.show(5)

+----------+------------+----------+--------+-------+--------+------------+------------------+
|CustomerID|CustomerName|   Product|Quantity|  Price|Discount|PurchaseDate|purchaseDatestring|
+----------+------------+----------+--------+-------+--------+------------+------------------+
|      1040|       Frank|Headphones|     5.0|30000.0|    NULL|  2024-01-01|        2024-01-01|
|      1007|       Diana|     Phone|     4.0|40000.0|    NULL|  2024-01-02|        2024-01-02|
|      1001|       Heidi|      NULL|     5.0|40000.0|     0.0|  2024-01-03|        2024-01-03|
|      1047|       Grace|     Phone|    NULL|40000.0|    15.0|  2024-01-04|        2024-01-04|
|      1017|       Heidi|    Laptop|     2.0|20000.0|    NULL|  2024-01-05|        2024-01-05|
+----------+------------+----------+--------+-------+--------+------------+------------------+
only showing top 5 rows



In [0]:
'''literal means a fixed constant value that we want to use inside a DataFrame expression.

    Since DF works with columns (not normal python variables), we cannot directly use constants like 5 or 'abc' in columnn expressions.
    
    Instead wrap them with lit() so pyspark knows it's a constant column value'''

from pyspark.sql.functions import lit

df_14 = df.withColumn('Country', lit('USA'))
df_14.show(5)

# df.withColumn("id_plus_10", col("id") + 10).show() -> it works without lit(), but safer side use lit

+----------+------------+----------+--------+-------+--------+------------+-------+
|CustomerID|CustomerName|   Product|Quantity|  Price|Discount|PurchaseDate|Country|
+----------+------------+----------+--------+-------+--------+------------+-------+
|      1040|       Frank|Headphones|     5.0|30000.0|    NULL|  2024-01-01|    USA|
|      1007|       Diana|     Phone|     4.0|40000.0|    NULL|  2024-01-02|    USA|
|      1001|       Heidi|      NULL|     5.0|40000.0|     0.0|  2024-01-03|    USA|
|      1047|       Grace|     Phone|    NULL|40000.0|    15.0|  2024-01-04|    USA|
|      1017|       Heidi|    Laptop|     2.0|20000.0|    NULL|  2024-01-05|    USA|
+----------+------------+----------+--------+-------+--------+------------+-------+
only showing top 5 rows



In [0]:
df_15 = df.groupBy('CustomerId').pivot('Product').agg(sum(col('Price')))
df_15.show(5)

+----------+-------+----------+--------+-------+-------+-------+-----+-------+
|CustomerId|   null|Headphones|Keyboard| Laptop|Monitor|  Mouse|Phone| Tablet|
+----------+-------+----------+--------+-------+-------+-------+-----+-------+
|      1025|   NULL|      NULL| 30000.0|   NULL|   NULL|   NULL| NULL|   NULL|
|      1005|40000.0|      NULL| 40000.0|   NULL|   NULL|30000.0| NULL|   NULL|
|      1016|   NULL|      NULL|    NULL|   NULL|   NULL|   NULL| NULL|   NULL|
|      1034|   NULL|   30000.0| 40000.0|40000.0|   NULL|   NULL| NULL|   NULL|
|      1046|   NULL|      NULL|    NULL|   NULL|40000.0|   NULL| NULL|40000.0|
+----------+-------+----------+--------+-------+-------+-------+-----+-------+
only showing top 5 rows



In [0]:
from pyspark.sql.functions import current_date, current_timestamp

df_16 = df.withColumn('current date', current_date())\
          .withColumn('current time stamp', current_timestamp())
df_16.show(5)

+----------+------------+----------+--------+-------+--------+------------+------------+--------------------+
|CustomerID|CustomerName|   Product|Quantity|  Price|Discount|PurchaseDate|current date|  current time stamp|
+----------+------------+----------+--------+-------+--------+------------+------------+--------------------+
|      1040|       Frank|Headphones|     5.0|30000.0|    NULL|  2024-01-01|  2025-08-27|2025-08-27 00:30:...|
|      1007|       Diana|     Phone|     4.0|40000.0|    NULL|  2024-01-02|  2025-08-27|2025-08-27 00:30:...|
|      1001|       Heidi|      NULL|     5.0|40000.0|     0.0|  2024-01-03|  2025-08-27|2025-08-27 00:30:...|
|      1047|       Grace|     Phone|    NULL|40000.0|    15.0|  2024-01-04|  2025-08-27|2025-08-27 00:30:...|
|      1017|       Heidi|    Laptop|     2.0|20000.0|    NULL|  2024-01-05|  2025-08-27|2025-08-27 00:30:...|
+----------+------------+----------+--------+-------+--------+------------+------------+--------------------+
only showi

In [0]:
''' replace() is similar to fillna() but it replaces non null values'''

df_17 = df.replace({'Headphones':'Headset','Phone':'MobilePhone'},subset='Product')
df_17.show(5)

# If subset - column names are not mentioned, it replaces everywhere in the dataframe

+----------+------------+-----------+--------+-------+--------+------------+
|CustomerID|CustomerName|    Product|Quantity|  Price|Discount|PurchaseDate|
+----------+------------+-----------+--------+-------+--------+------------+
|      1040|       Frank|    Headset|     5.0|30000.0|    NULL|  2024-01-01|
|      1007|       Diana|MobilePhone|     4.0|40000.0|    NULL|  2024-01-02|
|      1001|       Heidi|       NULL|     5.0|40000.0|     0.0|  2024-01-03|
|      1047|       Grace|MobilePhone|    NULL|40000.0|    15.0|  2024-01-04|
|      1017|       Heidi|     Laptop|     2.0|20000.0|    NULL|  2024-01-05|
+----------+------------+-----------+--------+-------+--------+------------+
only showing top 5 rows



In [0]:
'drop rows with NULL values'

'''Syntax :- df.na.drop(subset=[], how='any', thres=None)'''

df_18 = df.na.drop(subset=['Product','Quantity'])
df_18.show(5)

+----------+------------+----------+--------+-------+--------+------------+
|CustomerID|CustomerName|   Product|Quantity|  Price|Discount|PurchaseDate|
+----------+------------+----------+--------+-------+--------+------------+
|      1040|       Frank|Headphones|     5.0|30000.0|    NULL|  2024-01-01|
|      1007|       Diana|     Phone|     4.0|40000.0|    NULL|  2024-01-02|
|      1017|       Heidi|    Laptop|     2.0|20000.0|    NULL|  2024-01-05|
|      1015|     Charlie|    Laptop|     2.0|   NULL|    NULL|  2024-01-06|
|      1014|         Eve|     Phone|     2.0|40000.0|     0.0|  2024-01-07|
+----------+------------+----------+--------+-------+--------+------------+
only showing top 5 rows

